
# Left and Right Inverses of a Matrix 



## Definitions
- A **left inverse** of $A$ is a matrix $L$ such that $LA = I$. It exists when $A$ has **full column rank** (e.g., a *tall* matrix, more rows than columns).
- A **right inverse** of $A$ is a matrix $R$ such that $AR = I$. It exists when $A$ has **full row rank** (e.g., a *wide* matrix, more columns than rows).
- When $A$ is square and invertible, $A^{-1}$ is both a left and a right inverse.

**Least squares (tall matrices):** For full column rank, the canonical left inverse is $L = (A^\top A)^{-1}A^\top$. Then the least‑squares solution to $Ax = b$ is $x = Lb$, and $Ax$ is the orthogonal **projection of $b$ onto Col(A)**.

**Minimum norm (wide matrices):** For full row rank, a right inverse is $R = A^\top(AA^\top)^{-1}$. Then one solution of $Ax = b$ is $x = Rb$. 


 


When to prefer computing the right inverse directly?

If A is $m\times n$ with $m≪n$, computing the right inverse via
$R=A^⊤(AA^⊤)^{−1}$
requires inverting only an $m\times m$ matrix $AA^⊤$, which is much cheaper than anything that touches an $n \times n$ object.




Example scenario:
$m=100$, $n=100,000$.

$AA^⊤$ is $100\times 100$ →  tractable.
$A^⊤A$ is $100,000\times 100,000$ → infeasible.

In [1]:

import numpy as np

def is_close(A, B, tol=1e-9):
    return np.allclose(A, B, atol=tol, rtol=0)

def print_matrix(name, M):
    print(f"{name} shape={M.shape} {M} ")

def orthogonality_check(A, r, tol=1e-9):
    v = A.T @ r
    print(f"||A^T r||_2 = {np.linalg.norm(v):.3e} (should be ~ 0 if residual is orthogonal to col(A)) ")




## 1) Tall matrix (full column rank) → Left inverse & least squares geometry
**Theory:** If $A\in\mathbb{R}^{m\times n}$ with $m>n$ and full column rank, then $L=(A^\top A)^{-1}A^\top$ satisfies $LA=I_n$. For any $b\in\mathbb{R}^m$, the least‑squares solution is $x=Lb$. The fitted vector $\hat b=Ax$ is the **orthogonal projection** of $b$ onto $col(A)$, and the residual $r=b-\hat b$ is orthogonal to that space (i.e., $A^\top r=0$).


In [2]:

# A simple tall matrix (3x2) with full column rank
A_tall = np.array([[1., 2.],
                   [3., 4.],
                   [5., 7.]])

# Left inverse via normal equations 
L = np.linalg.inv(A_tall.T @ A_tall) @ A_tall.T

print_matrix('A_tall', A_tall)
print_matrix('Left inverse L = (A^T A)^{-1} A^T', L)
print('Check L A = I: ', is_close(L @ A_tall, np.eye(A_tall.shape[1])))

# Least squares for Ax = b
b = np.array([2., 1., -1.])
x_ls = L @ b
b_hat = A_tall @ x_ls
r = b - b_hat

print_matrix('b', b)
print_matrix('x_ls (least squares)', x_ls)
print_matrix('b_hat = A x_ls (projection of b onto col(A))', b_hat)
print_matrix('Residual r = b - b_hat', r)
orthogonality_check(A_tall, r)


A_tall shape=(3, 2) [[1. 2.]
 [3. 4.]
 [5. 7.]] 
Left inverse L = (A^T A)^{-1} A^T shape=(2, 3) [[-2.07142857e+00  7.85714286e-01  1.42857143e-01]
 [ 1.50000000e+00 -5.00000000e-01  3.55271368e-15]] 
Check L A = I:  True
b shape=(3,) [ 2.  1. -1.] 
x_ls (least squares) shape=(2,) [-3.5  2.5] 
b_hat = A x_ls (projection of b onto col(A)) shape=(3,) [ 1.50000000e+00 -5.00000000e-01 -7.10542736e-15] 
Residual r = b - b_hat shape=(3,) [ 0.5  1.5 -1. ] 
||A^T r||_2 = 8.967e-14 (should be ~ 0 if residual is orthogonal to col(A)) 



## 2) Wide matrix (full row rank) → Right inverse 
**Theory:** If $A\in\mathbb{R}^{m\times n}$ with $m<n$ and full row rank, then $R=A^\top(AA^\top)^{-1}$ satisfies $AR=I_m$. Given $b\in\mathbb{R}^m$, one solution to $Ax=b$ is $x=Rb$.


In [3]:
# A simple wide matrix (2x3) with full row rank
A_wide = np.array([[1., 2., 3.],
                  [2., 1., 0.]])

# Right inverse via normal equations on rows 
R = A_wide.T @ np.linalg.inv(A_wide @ A_wide.T)

print_matrix('A_wide', A_wide)
print_matrix('Right inverse R = A^T (A A^T)^{-1}', R)
print('Check A R = I: ', is_close(A_wide @ R, np.eye(A_wide.shape[0])))



A_wide shape=(2, 3) [[1. 2. 3.]
 [2. 1. 0.]] 
Right inverse R = A^T (A A^T)^{-1} shape=(3, 2) [[-0.05555556  0.44444444]
 [ 0.11111111  0.11111111]
 [ 0.27777778 -0.22222222]] 
Check A R = I:  True



## 3) The transpose trick
**Key idea:** If $A$ is tall full‑column‑rank and you computed the left inverse $L=(A^\top A)^{-1}A^\top$, then for **the transposed problem** $A^\top$ (which is wide full‑row‑rank), the matrix **$L^\top$ is a right inverse of $A^\top$**, because

$$
(A^\top)\,L^\top = (LA)^\top = I^\top = I.
$$

This is handy if an algorithm needs both forms but building only one is cheaper or already available.


In [4]:

# Verify the transpose trick with our tall example
A = A_tall
L = np.linalg.inv(A.T @ A) @ A.T
check = A.T @ L.T
print_matrix('(A^T) * (L^T)', check)
print('Is it identity?', is_close(check, np.eye(A.shape[1])))


(A^T) * (L^T) shape=(2, 2) [[1.00000000e+00 1.77635684e-14]
 [8.88178420e-15 1.00000000e+00]] 
Is it identity? True


Exercises:
1. For a large tall matrix, find right inverse using left inverse (Hint: Use transpose idea). Verify your answer. 
Give an explicit illustration with $3 \times 2$ matrix.
2. For a large wide matrix, find left inverse using right invese. Verify your answer. Give an explicit illustration with $2 \times 3$ matrix.

In [6]:
A = np.array([[1., 2.],
              [3., 4.],
              [5., 7.]])

L = np.linalg.inv(A.T @ A) @ A.T
R = L.T

print("L =\n", L)
print("Right inverse of A.T =\n", R)
print("A.T @ R =\n", A.T @ R)

L =
 [[-2.07142857e+00  7.85714286e-01  1.42857143e-01]
 [ 1.50000000e+00 -5.00000000e-01  3.55271368e-15]]
Right inverse of A.T =
 [[-2.07142857e+00  1.50000000e+00]
 [ 7.85714286e-01 -5.00000000e-01]
 [ 1.42857143e-01  3.55271368e-15]]
A.T @ R =
 [[1.00000000e+00 1.77635684e-14]
 [8.88178420e-15 1.00000000e+00]]


In [7]:
A = np.array([[1., 2., 3.],
              [2., 1., 0.]])

R = A.T @ np.linalg.inv(A @ A.T)
L = R.T

print("R =\n", R)
print("Left inverse of A.T =\n", L)
print("L @ A.T =\n", L @ A.T)

R =
 [[-0.05555556  0.44444444]
 [ 0.11111111  0.11111111]
 [ 0.27777778 -0.22222222]]
Left inverse of A.T =
 [[-0.05555556  0.11111111  0.27777778]
 [ 0.44444444  0.11111111 -0.22222222]]
L @ A.T =
 [[1. 0.]
 [0. 1.]]
